In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import urllib.request

In [3]:
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [4]:
with open("the-verdict.txt", "r", encoding="UTF-8") as f:
  raw_text = f.read()

In [5]:
len(raw_text)

20479

In [6]:
import re

In [7]:
text = "Hello world, what happen"
print(text.split())

['Hello', 'world,', 'what', 'happen']


In [8]:
text = "Hello world, what happen"
result = re.split(r"(\s)", text)
print(result)

['Hello', ' ', 'world,', ' ', 'what', ' ', 'happen']


In [9]:
## Seperate punctuation

result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ' ', 'world', ',', '', ' ', 'what', ' ', 'happen']


In [10]:
result = [item for item in result if item.strip()]
print(result)

['Hello', 'world', ',', 'what', 'happen']


In [11]:
text = "Hello world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item for item in result if item.strip()]

In [12]:
len(result)

4690

In [13]:
result[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

In [14]:
## Unique Word

all_words = sorted(set(result))
vocab_size = len(all_words)
print(vocab_size)

1130


In [15]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [16]:
class SimpleTokenizerV1:

  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}

  def encode_text(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

    preprocessed = [
      item.strip() for item in preprocessed if item.strip()
    ]

    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    # Replace spaces before the specified punctuations
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [17]:
vocab["HAD"]

44

In [18]:
int_to_str = {i:s for s, i in vocab.items()}
int_to_str[44]

'HAD'

In [19]:
tokeinzed = SimpleTokenizerV1(vocab)

In [20]:
text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""

ids = tokeinzed.encode_text(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [21]:
string = tokeinzed.decode(ids)
print(string)

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [22]:
text = "hello, I would prefer coffee over tea"
tokeinzed.encode_text(text)

KeyError: 'hello'

In [23]:
all_words = sorted(list(set(result)))
all_words.extend(["<|endoftext|>", "<|UNK|>"])

In [24]:
len(all_words)

1132

In [25]:
print(all_words[-5:])

['younger', 'your', 'yourself', '<|endoftext|>', '<|UNK|>']


In [26]:
vocab = {token:ids for ids, token in enumerate(all_words)}

In [27]:
class SimpleTokenizerV2:

  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}

  def encode_text(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

    preprocessed = [
            item.strip() for item in preprocessed if item.strip()
      ]

    preprocessed = [
        item if item in self.str_to_int else "<|UNK|>" for item in preprocessed
    ]

    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    # Replace spaces before the specified punctuations
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [28]:
tokenv2 = SimpleTokenizerV2(vocab)

text = "Hello, I would prefer coffee over tea"
ids = tokenv2.encode_text(text)
ids

[1131, 5, 53, 1120, 1131, 1131, 741, 975]

In [29]:
string = tokenv2.decode(ids=ids)
print(string)

<|UNK|>, I would <|UNK|> <|UNK|> over tea


## Bytepair encoding
- It breaks down the longer unknown word into individual sub word

In [30]:
import tiktoken

In [31]:
tiktoken.__version__

'0.14.0'

In [32]:
tokenizer = tiktoken.get_encoding("gpt2")

In [33]:
encode = tokenizer.encode("Hello, i am world")
encode

[15496, 11, 1312, 716, 995]

In [34]:
tokenizer.decode(encode)

'Hello, i am world'

### Data sampling with sliding window

In [35]:
with open("the-verdict.txt", 'r') as f:
    text = f.read()

enc_text = tokenizer.encode(text)
print(len(enc_text))

5145


In [36]:
enc_sample = enc_text[50:]
print(len(enc_sample))
print(len(enc_text))

5095
5145


In [37]:
context_size = 5

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print("x: ", x)
print("y:     ", y)

x:  [290, 4920, 2241, 287, 257]
y:      [4920, 2241, 287, 257, 4489]


In [38]:
for i in range(1, context_size):
    context = enc_sample[:i]
    desire = enc_sample[i]

    print(tokenizer.decode(context), "-->", tokenizer.decode([desire]))

 and -->  established
 and established -->  himself
 and established himself -->  in
 and established himself in -->  a


In [39]:
import torch

In [40]:
torch.__version__

'2.14.0'

In [41]:
from torch.utils.data import Dataset, DataLoader

In [42]:
class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):
        super().__init__()

        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        ## Implementing sliding window
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length]
            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return torch.tensor(self.input_ids[index]), torch.tensor(self.target_ids[index])

In [43]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding('gpt2')

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloder = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle=shuffle,
        drop_last = drop_last,
        num_workers=num_workers
    )
    return dataloder

In [44]:
dataloader = create_dataloader_v1(text, batch_size=1, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464]])]

In [45]:
second_batch = next(data_iter)
second_batch

[tensor([[1807, 3619,  402,  271]]), tensor([[3619,  402,  271]])]

In [46]:
dataloader = create_dataloader_v1(text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
input, target = next(data_iter)
print("Input: \n", input)
print("target: \n", target)

Input: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
target: 
 tensor([[ 367, 2885, 1464],
        [3619,  402,  271],
        [2138,  257, 7026],
        [ 438, 2016,  257],
        [5891, 1576,  438],
        [ 340,  373,  645],
        [5975,  284,  502],
        [3285,  326,   11]])


### Creating token embedding

In [47]:
input = torch.tensor([1, 2, 3, 4])

In [48]:
vocab = 6
dim = 3

torch.manual_seed(123)
embedding = torch.nn.Embedding(vocab, dim)

In [49]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

### Encoding word position

In [50]:
vocab_size = tokenizer.n_vocab
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [51]:
max_lenght = 4
dataloader = create_dataloader_v1(text, batch_size=8, max_length=max_lenght, stride=max_lenght, shuffle=False)
data_iter = iter(dataloader)
input, target = next(data_iter)
print("Input: \n", input)
print("target: \n", target)

Input: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
target: 
 tensor([[ 367, 2885, 1464],
        [3619,  402,  271],
        [2138,  257, 7026],
        [ 438, 2016,  257],
        [5891, 1576,  438],
        [ 340,  373,  645],
        [5975,  284,  502],
        [3285,  326,   11]])


In [52]:
token_embedding = token_embedding_layer(input)
token_embedding.shape

torch.Size([8, 4, 256])

In [53]:
context_lenght = max_lenght
pos_embedding_layer = torch.nn.Embedding(context_lenght, output_dim)

In [54]:
pos_embedding_layer.weight

Parameter containing:
tensor([[ 1.7375, -0.5620, -0.6303,  ..., -0.2277,  1.5748,  1.0345],
        [ 1.6423, -0.7201,  0.2062,  ...,  0.4118,  0.1498, -0.4628],
        [-0.4651, -0.7757,  0.5806,  ...,  1.4335, -0.4963,  0.8579],
        [-0.6754, -0.4628,  1.4323,  ...,  0.8139, -0.7088,  0.4827]],
       requires_grad=True)

In [55]:
pos_embedding = pos_embedding_layer(torch.arange(max_lenght))
pos_embedding

tensor([[ 1.7375, -0.5620, -0.6303,  ..., -0.2277,  1.5748,  1.0345],
        [ 1.6423, -0.7201,  0.2062,  ...,  0.4118,  0.1498, -0.4628],
        [-0.4651, -0.7757,  0.5806,  ...,  1.4335, -0.4963,  0.8579],
        [-0.6754, -0.4628,  1.4323,  ...,  0.8139, -0.7088,  0.4827]],
       grad_fn=<EmbeddingBackward0>)

In [56]:
pos_embedding.shape

torch.Size([4, 256])

In [57]:
token_embedding.shape

torch.Size([8, 4, 256])

In [58]:
input_embedding = token_embedding + pos_embedding

In [59]:
input_embedding.shape

torch.Size([8, 4, 256])

### Coding Attention mechanism